# 2C · Mini-Project: Portfolio Position Tracker
### Financial Analytics — Module 2

Everything from 2A and 2B, combined into one working tool — **in pure Python, no libraries.**

You'll build a tracker that:
1. Stores a portfolio of Indian stocks with buy prices
2. Values it at current market prices
3. Computes profit/loss per holding and overall
4. Flags positions breaching a stop-loss
5. Prints a clean report

This is genuinely useful code. It's also the *manual* version of what pandas will do in one line in Module 3 — doing it by hand first is what makes Module 3 make sense.

---
## Step 1 — The data

Each holding needs: quantity, buy price, current price. A **dict of dicts** handles this cleanly.

In [ ]:
portfolio = {
    "RELIANCE.NS":  {"qty": 150, "buy_price": 2380.00, "current": 2512.40},
    "TCS.NS":       {"qty":  95, "buy_price": 3620.50, "current": 3450.75},
    "HDFCBANK.NS":  {"qty": 200, "buy_price": 1445.20, "current": 1520.40},
    "INFY.NS":      {"qty": 120, "buy_price": 1530.00, "current": 1480.50},
    "ITC.NS":       {"qty": 500, "buy_price":  398.75, "current":  415.20},
}

# Look inside one holding
print(portfolio["TCS.NS"])
print("TCS quantity:", portfolio["TCS.NS"]["qty"])

---
## Step 2 — Functions for the calculations

One function per idea. Each gets a tracer-bullet test.

In [ ]:
def cost_basis(holding):
    """What we paid, in rupees."""
    return holding["qty"] * holding["buy_price"]


def market_value(holding):
    """What it's worth now, in rupees."""
    return holding["qty"] * holding["current"]


def pnl(holding):
    """Absolute profit or loss, in rupees."""
    return market_value(holding) - cost_basis(holding)


def pnl_pct(holding):
    """Profit or loss as a percentage of cost."""
    return pnl(holding) / cost_basis(holding) * 100


# TRACER BULLETS - test each on numbers you can verify in your head
test = {"qty": 10, "buy_price": 100.0, "current": 110.0}
print("cost_basis   expect 1000 ->", cost_basis(test))
print("market_value expect 1100 ->", market_value(test))
print("pnl          expect  100 ->", pnl(test))
print("pnl_pct      expect   10 ->", pnl_pct(test))

---
## Step 3 — The report

A loop over the portfolio, with formatting. `:>12,.2f` means: right-align, width 12, comma separators, 2 decimals.

In [ ]:
def print_report(portfolio):
    print(f"{'TICKER':<14}{'QTY':>6}{'BUY':>11}{'NOW':>11}{'VALUE':>14}{'P&L':>13}{'P&L %':>9}")
    print("=" * 78)

    total_cost = 0
    total_value = 0

    for ticker, h in portfolio.items():
        total_cost += cost_basis(h)
        total_value += market_value(h)
        print(f"{ticker:<14}{h['qty']:>6}{h['buy_price']:>11,.2f}{h['current']:>11,.2f}"
              f"{market_value(h):>14,.2f}{pnl(h):>13,.2f}{pnl_pct(h):>8.2f}%")

    total_pnl = total_value - total_cost
    total_pct = total_pnl / total_cost * 100

    print("=" * 78)
    print(f"{'PORTFOLIO':<14}{'':>6}{'':>11}{'':>11}{total_value:>14,.2f}{total_pnl:>13,.2f}{total_pct:>8.2f}%")
    print(f"\nInvested: Rs {total_cost:,.2f}   |   Current: Rs {total_value:,.2f}")


print_report(portfolio)

---
## Step 4 — Risk: stop-loss monitoring

A stop-loss is a rule: if a position falls more than X% below its buy price, flag it.

In [ ]:
def check_stops(portfolio, stop_pct=-5.0):
    """Return a list of tickers breaching the stop-loss threshold."""
    breaches = []
    for ticker, h in portfolio.items():
        if pnl_pct(h) <= stop_pct:
            breaches.append((ticker, round(pnl_pct(h), 2)))
    return breaches


alerts = check_stops(portfolio, stop_pct=-4.0)

if alerts:
    print("STOP-LOSS ALERTS (threshold -4%):")
    for ticker, loss in alerts:
        print(f"  {ticker:<14} {loss:>7.2f}%")
else:
    print("No positions breaching the stop loss.")

---
## Step 5 — Concentration risk

The classic first risk question: *how much of the portfolio sits in one name?* Anything above ~25% is a concentration flag in most mandates.

In [ ]:
def weights(portfolio):
    """Each holding's share of total portfolio value, as a percentage."""
    total = sum(market_value(h) for h in portfolio.values())   # comprehension inside sum()
    return {t: market_value(h) / total * 100 for t, h in portfolio.items()}


w = weights(portfolio)

# Sort by weight, largest first
for ticker, pct in sorted(w.items(), key=lambda pair: pair[1], reverse=True):
    bar = "#" * int(pct / 2)              # a poor man's bar chart
    flag = "  <-- CONCENTRATED" if pct > 25 else ""
    print(f"{ticker:<14}{pct:>6.2f}%  {bar}{flag}")

print(f"\nWeights sum to {sum(w.values()):.1f}% (sanity check - should be 100)")

---
## ✏️ Your turn — three extensions

Complete these. They use only what Modules 2A and 2B taught.

### Exercise 1 — Add a holding
Write `add_holding(portfolio, ticker, qty, buy_price, current)` that adds a new position to the dict. Add SBIN.NS: 300 shares, bought at Rs 610.00, now Rs 590.25. Then re-run the report.

In [ ]:
def add_holding(portfolio, ticker, qty, buy_price, current):
    # your code here
    pass


add_holding(portfolio, "SBIN.NS", 300, 610.00, 590.25)
print_report(portfolio)

### Exercise 2 — Best and worst
Write `best_and_worst(portfolio)` that returns a tuple: `(best_ticker, worst_ticker)` by percentage P&L.

In [ ]:
def best_and_worst(portfolio):
    # your code here
    pass


print(best_and_worst(portfolio))

### Exercise 3 — Sell a position
Write `sell(portfolio, ticker, qty)` that reduces the quantity — and removes the holding entirely if quantity reaches zero. Sell 100 ITC shares, then all 120 INFY shares, then print the report.

In [ ]:
def sell(portfolio, ticker, qty):
    # your code here
    pass


sell(portfolio, "ITC.NS", 100)
sell(portfolio, "INFY.NS", 120)
print_report(portfolio)

---
## What you just built — and what comes next

You wrote a functioning portfolio tracker in pure Python: valuation, P&L, risk alerts, concentration analysis. No libraries.

**Now the honest part.** In Module 3, this entire notebook collapses into a few lines:

```python
df["value"] = df["qty"] * df["current"]
df["pnl"]   = df["value"] - df["qty"] * df["buy_price"]
df["weight"] = df["value"] / df["value"].sum() * 100
df.sort_values("weight", ascending=False)
```

That is pandas. It will feel like magic — but only because you now know exactly what it's doing underneath. Learners who skip straight to pandas never get that.

**Module 2 complete.** Claim your **Speaks Python** badge.

---
*AI disclosure: did AI help you here? Note what for: ______*